# Early-Stage Diabetes Risk Prediction

## 3. Data Preprocessing

This notebook prepares the dataset for machine learning while keeping preprocessing steps separate from model training. The workflow includes data validation, missing-value and duplicate checks, target preparation, leakage checks, categorical encoding, numerical scaling, and a train/test split.

### 3.1 Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

### 3.2 Load the Dataset

In [2]:
df = pd.read_csv("../data/diabetes_dataset.csv")

print("Initial dataset shape:", df.shape)
df.head()

Initial dataset shape: (100000, 31)


,age,gender,ethnicity,education_level,income_level,employment_status,smoking_status,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,...,hdl_cholesterol,ldl_cholesterol,triglycerides,glucose_fasting,glucose_postprandial,insulin_level,hba1c,diabetes_risk_score,diabetes_stage,diagnosed_diabetes
0,58,Male,Asian,Highschool,Lower-Middle,Employed,Never,0,215,5.7,...,41,160,145,136,236,6.36,8.18,29.6,Type 2,1
1,48,Female,White,Highschool,Middle,Employed,Former,1,143,6.7,...,55,50,30,93,150,2.00,5.63,23.0,No Diabetes,0
2,60,Male,Hispanic,Highschool,Middle,Unemployed,Never,1,57,6.4,...,66,99,36,118,195,5.07,7.51,44.7,Type 2,1
3,74,Female,Black,Highschool,Low,Retired,Never,0,49,3.4,...,50,79,140,139,253,5.28,9.03,38.2,Type 2,1
4,46,Male,White,Graduate,Middle,Retired,Never,1,109,7.2,...,52,125,160,137,184,12.74,7.20,23.5,Type 2,1


### 3.3 Data Inspection

In [3]:
print("Data types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Data types:


age                                     int64
gender                                    str
ethnicity                                 str
education_level                           str
income_level                              str
employment_status                         str
smoking_status                            str
alcohol_consumption_per_week            int64
physical_activity_minutes_per_week      int64
diet_score                            float64
sleep_hours_per_day                   float64
screen_time_hours_per_day             float64
family_history_diabetes                 int64
hypertension_history                    int64
cardiovascular_history                  int64
bmi                                   float64
waist_to_hip_ratio                    float64
systolic_bp                             int64
diastolic_bp                            int64
heart_rate                              int64
cholesterol_total                       int64
hdl_cholesterol                   


Missing values:


age                                   0
gender                                0
ethnicity                             0
education_level                       0
income_level                          0
employment_status                     0
smoking_status                        0
alcohol_consumption_per_week          0
physical_activity_minutes_per_week    0
diet_score                            0
sleep_hours_per_day                   0
screen_time_hours_per_day             0
family_history_diabetes               0
hypertension_history                  0
cardiovascular_history                0
bmi                                   0
waist_to_hip_ratio                    0
systolic_bp                           0
diastolic_bp                          0
heart_rate                            0
cholesterol_total                     0
hdl_cholesterol                       0
ldl_cholesterol                       0
triglycerides                         0
glucose_fasting                       0



Duplicate rows: 0


### 3.4 Duplicate Analysis and Removal

The dataset is checked for duplicate rows. If duplicates are present, they are removed before model development.

In [4]:
duplicates_before = df.duplicated().sum()

df = df.drop_duplicates().reset_index(drop=True)

print("Duplicate rows before removal:", duplicates_before)
print("Duplicate rows after removal:", df.duplicated().sum())
print("Dataset shape after duplicate removal:", df.shape)

Duplicate rows before removal: 0
Duplicate rows after removal: 0
Dataset shape after duplicate removal: (100000, 31)


### 3.5 Missing-Value Analysis

Missing values are checked before preprocessing. The preprocessing pipeline also contains median imputation for numerical variables and most-frequent imputation for categorical variables. Since our current dataset contains no missing values, these imputers should not change the observed values, but keeping them in the pipeline makes the preprocessing robust for future unseen input.

In [5]:
missing_summary = df.isnull().sum().sort_values(ascending=False)

print("Total missing values:", missing_summary.sum())
display(missing_summary[missing_summary > 0])

if missing_summary.sum() == 0:
    print("No missing values are present in the dataset.")

Total missing values: 0


Series([], dtype: int64)

No missing values are present in the dataset.


### 3.6 Target Preparation

The target variable is `diagnosed_diabetes`:
- `0` = No Diabetes
- `1` = Diabetes

In [6]:
target_column = "diagnosed_diabetes"
leakage_columns = ["diabetes_stage"]

X = df.drop(columns=[target_column] + leakage_columns)
y = df[target_column].astype(int)

print("Target variable:", target_column)
print("Excluded leakage columns:", leakage_columns)
print("Input feature count:", X.shape[1])
print("Target classes:", sorted(y.unique()))

print("\nTarget distribution:")
display(y.value_counts().sort_index())

print("\nTarget percentage:")
display((y.value_counts(normalize=True).sort_index() * 100).round(3))

Target variable: diagnosed_diabetes
Excluded leakage columns: ['diabetes_stage']
Input feature count: 29
Target classes: [np.int64(0), np.int64(1)]

Target distribution:


diagnosed_diabetes
0    40002
1    59998
Name: count, dtype: int64


Target percentage:


diagnosed_diabetes
0    40.002
1    59.998
Name: proportion, dtype: float64

### 3.7 Check Potential Data Leakage

`diabetes_risk_score` and `diabetes_stage` are retained in the dataset as requested for this project. However, both variables are checked because they could potentially contain information derived from the diagnosis target. We will inspect their relationship with `diagnosed_diabetes` before final model interpretation.

This check is important because a feature derived from the target would create data leakage and unrealistically high model performance.

In [7]:
potential_leakage_columns = ["diabetes_risk_score", "diabetes_stage"]

for col in potential_leakage_columns:
    print(f"\n--- {col} ---")
    
    if pd.api.types.is_numeric_dtype(df[col]):
        print("Numerical feature")
        display(
            df.groupby("diagnosed_diabetes")[col]
            .agg(["min", "max", "mean", "std"])
            .round(3)
        )
    else:
        print("Categorical feature")
        display(
            pd.crosstab(
                df[col],
                df["diagnosed_diabetes"],
                normalize="index"
            ).round(3)
        )


--- diabetes_risk_score ---
Numerical feature


,min,max,mean,std
diagnosed_diabetes,,,,
0,2.7,60.3,27.145,7.831
1,4.8,67.2,32.274,9.244



--- diabetes_stage ---
Categorical feature


diagnosed_diabetes,0,1
diabetes_stage,,
Gestational,0.432,0.568
No Diabetes,1.000,0.000
Pre-Diabetes,1.000,0.000
Type 1,0.459,0.541
Type 2,0.000,1.000


**Interpretation:** The above output is a leakage check, not a final decision based only on correlation or group averages. If either variable is found to be directly constructed from the diagnosis label, it should be excluded from the final modelling experiment and reported as a leakage-prevention decision.

### 3.8 Identify Feature Types

In [8]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print("Number of numerical features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

print("\nNumerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Number of numerical features: 23
Number of categorical features: 6

Numerical features:
['age', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week', 'diet_score', 'sleep_hours_per_day', 'screen_time_hours_per_day', 'family_history_diabetes', 'hypertension_history', 'cardiovascular_history', 'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol', 'triglycerides', 'glucose_fasting', 'glucose_postprandial', 'insulin_level', 'hba1c', 'diabetes_risk_score']

Categorical features:
['gender', 'ethnicity', 'education_level', 'income_level', 'employment_status', 'smoking_status']


### 3.9 Train/Test Split

The data is split before fitting the preprocessing transformations. This prevents information from the test set from being used to learn the encoder or scaler.

A stratified split is used so that the target-class proportions are maintained in both sets.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)
print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

print("\nTraining target distribution:")
display((y_train.value_counts(normalize=True).sort_index() * 100).round(3))

print("\nTesting target distribution:")
display((y_test.value_counts(normalize=True).sort_index() * 100).round(3))

Training feature shape: (80000, 29)
Testing feature shape: (20000, 29)
Training target shape: (80000,)
Testing target shape: (20000,)

Training target distribution:


diagnosed_diabetes
0    40.003
1    59.998
Name: proportion, dtype: float64


Testing target distribution:


diagnosed_diabetes
0    40.0
1    60.0
Name: proportion, dtype: float64

### 3.10 Build the Preprocessing Pipeline

In [10]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


### 3.11 Fit Preprocessing Only on Training Data

The preprocessing transformations are fitted using **training data only**. The already-fitted transformations are then applied to the test data.

This is the main evidence that preprocessing is performed without data leakage.

In [11]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (80000, 47)
Processed testing shape: (20000, 47)


### 3.12 Get the Final Processed Feature Names

In [12]:
processed_feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(processed_feature_names))
display(pd.DataFrame({"Processed Feature": processed_feature_names}))

Number of processed features: 47


,Processed Feature
0,numeric__age
1,numeric__alcohol_consumption_per_week
2,numeric__physical_activity_minutes_per_week
3,numeric__diet_score
4,numeric__sleep_hours_per_day
5,numeric__screen_time_hours_per_day
6,numeric__family_history_diabetes
7,numeric__hypertension_history
8,numeric__cardiovascular_history
9,numeric__bmi


### 3.13 Check Scaling of Numerical Features

In [13]:
numeric_feature_indices = preprocessor.transformers_[0][2]

print("Number of original numerical features:", len(numeric_feature_indices))

# The first part of the transformed matrix corresponds to the numerical pipeline.
scaled_numeric = X_train_processed[:, :len(numeric_features)]

print("Mean of scaled numerical features (training data):")
print(np.round(scaled_numeric.mean(axis=0), 3))

print("\nStandard deviation of scaled numerical features (training data):")
print(np.round(scaled_numeric.std(axis=0), 3))

Number of original numerical features: 23
Mean of scaled numerical features (training data):
[ 0.  0. -0.  0. -0.  0.  0. -0. -0.  0.  0. -0. -0. -0. -0.  0. -0.  0.
 -0. -0.  0. -0.  0.]

Standard deviation of scaled numerical features (training data):
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


### 3.14 Preprocessing Summary

In [14]:
print("PREPROCESSING SUMMARY")
print("=" * 50)
print("Initial dataset shape:       ", df.shape)
print("Duplicate rows originally:  ", duplicates_before)
print("Missing values:              ", df.isnull().sum().sum())
print("Input features before encoding:", X.shape[1])
print("Original numerical features: ", len(numeric_features))
print("Original categorical features:", len(categorical_features))
print("Training samples:            ", X_train.shape[0])
print("Testing samples:             ", X_test.shape[0])
print("Processed training features:", X_train_processed.shape[1])
print("Processed testing features: ", X_test_processed.shape[1])
print("Target variable:             ", target_column)

PREPROCESSING SUMMARY
Initial dataset shape:        (100000, 31)
Duplicate rows originally:   0
Missing values:               0
Input features before encoding: 29
Original numerical features:  23
Original categorical features: 6
Training samples:             80000
Testing samples:              20000
Processed training features: 47
Processed testing features:  47
Target variable:              diagnosed_diabetes


In [15]:
print("Unique diabetes stages:")
print(df["diabetes_stage"].value_counts())

print("\nDiabetes stage vs diagnosed diabetes:")
stage_target = pd.crosstab(
    df["diabetes_stage"],
    df["diagnosed_diabetes"]
)

display(stage_target)

Unique diabetes stages:
diabetes_stage
Type 2          59774
Pre-Diabetes    31845
No Diabetes      7981
Gestational       278
Type 1            122
Name: count, dtype: int64

Diabetes stage vs diagnosed diabetes:


diagnosed_diabetes,0,1
diabetes_stage,,
Gestational,120,158
No Diabetes,7981,0
Pre-Diabetes,31845,0
Type 1,56,66
Type 2,0,59774


In [16]:
print("Diabetes risk score statistics by diagnosis:")

risk_score_analysis = df.groupby("diagnosed_diabetes")["diabetes_risk_score"].agg(
    ["min", "max", "mean", "std"]
).round(3)

display(risk_score_analysis)

Diabetes risk score statistics by diagnosis:


,min,max,mean,std
diagnosed_diabetes,,,,
0,2.7,60.3,27.145,7.831
1,4.8,67.2,32.274,9.244


## Preprocessing Conclusions

The dataset was inspected for missing values and duplicates, the target variable was separated, and the data was split into training and testing sets using stratification. Numerical variables were standardized and categorical variables were one-hot encoded. The preprocessing transformations were fitted only on the training data and then applied to the test data, reducing the risk of data leakage.

The processed training and testing arrays are now ready for the baseline Random Forest and XGBoost experiments.